# 02 — Feature Engineering

This notebook constructs the **item feature matrix** used by the content-based
and hybrid recommender models.

Starting from the processed movies table produced in notebook 01, we build two
complementary feature blocks: (1) a **multi-hot genre indicator** vector and
(2) a **TF-IDF representation** over the concatenated movie title and aggregated
tag text, compressed into 256 dimensions via TruncatedSVD (Latent Semantic Analysis).
The resulting sparse matrix and all fitted transformers are persisted to disk so that
the exact same feature space is applied at inference time without re-fitting.

**Steps:**
Load processed data → Genre matrix → TF-IDF on title + tags →
TruncatedSVD compression → Assemble & save item feature matrix.


In [1]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.features import (
    build_genre_matrix, build_text_matrix,
    build_item_features, save_item_features,
)

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1200, height=600, scale=2)
    except Exception:
        pass  # kaleido not installed — HTML only
    fig.show()


## 1. Load Processed Data

In [2]:
movies = load_processed("movies")
print(f"Movies loaded: {len(movies):,}")
print(f"Movies with tags: {(movies['tags_text'] != '').sum():,} "
      f"({100 * (movies['tags_text'] != '').mean():.1f}%)")
display(movies[["movieId", "clean_title", "year", "genres", "tags_text"]].head(5))

print(f"\nYear parsed for {movies['year'].notna().mean() * 100:.1f}% of movies")


Movies loaded: 62,423
Movies with tags: 45,251 (72.5%)


,movieId,clean_title,year,genres,tags_text
0,1,Toy Story,1995,Adventure|Animation|Children|Comedy|Fantasy,owned imdb top 250 pixar pixar time travel chi...
1,2,Jumanji,1995,Adventure|Children|Fantasy,robin williams time travel fantasy based on ch...
2,3,Grumpier Old Men,1995,Comedy|Romance,funny best friend duringcreditsstinger fishing...
3,4,Waiting to Exhale,1995,Comedy|Drama|Romance,based on novel or book chick flick divorce int...
4,5,Father of the Bride Part II,1995,Comedy,aging baby confidence contraception daughter g...



Year parsed for 99.1% of movies


### Tag-text richness

The content text block is built from each movie's aggregated tags. How much text is
there per movie? Movies with no tags contribute only their title — so this distribution
tells us how strong the textual signal is across the catalogue.


In [3]:
word_count = movies["tags_text"].str.split().apply(len)
print("Tag-text length (words per movie):")
print(word_count.describe().round(1).to_string())

fig = px.histogram(
    word_count[word_count > 0], nbins=60, log_y=True,
    title="Tag-text Length (words) per Movie — tagged movies only",
    labels={"value": "Words of tag text", "count": "Number of movies"},
    color_discrete_sequence=["#AB63FA"],
)
fig.update_layout(showlegend=False)
save_fig(fig, "14_tagtext_wordcount")


Tag-text length (words per movie):
count    62423.0
mean        27.7
std        144.6
min          0.0
25%          0.0
50%          4.0
75%         13.0
max      10578.0


## 2. Genre Multi-hot Matrix

Each movie's pipe-separated genre string is expanded into a binary indicator
vector — one dimension per unique genre. This gives an interpretable, sparse
feature block that captures coarse content similarity at zero training cost.


In [4]:
genre_X     = build_genre_matrix(movies)
genre_names = movies["genres"].str.get_dummies(sep="|").columns.tolist()

print(f"Genre matrix shape: {genre_X.shape}")
print(f"Genres ({len(genre_names)}): {genre_names}")

genre_freq = (
    pd.DataFrame({
        "genre": genre_names,
        "count": np.asarray(genre_X.sum(axis=0)).ravel().astype(int),
    })
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
fig = px.bar(
    genre_freq, x="genre", y="count",
    title="Movies per Genre (multi-hot column sums)",
    color="count", color_continuous_scale="Teal",
)
fig.update_layout(xaxis_tickangle=-40, coloraxis_showscale=False)
save_fig(fig, "13_genre_frequency")
display(genre_freq)


Genre matrix shape: (62423, 20)
Genres (20): ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,genre,count
0,Drama,25606
1,Comedy,16870
2,Thriller,8654
3,Romance,7719
4,Action,7348
5,Horror,5989
6,Documentary,5605
7,Crime,5319
8,(no genres listed),5062
9,Adventure,4145


## 3. TF-IDF on Title + Tags

We concatenate the cleaned movie title with its aggregated tag text into one
document per movie, then apply **TF-IDF** (bigrams, sublinear TF, min_df = 5).
This encodes both the film's name semantics and the community's vocabulary around it,
producing a rich sparse representation of item content.


In [5]:
text_X, tfidf, svd_transformer = build_text_matrix(movies, n_components=256, min_df=5)

print(f"TF-IDF vocabulary size:       {len(tfidf.vocabulary_):,} terms")
print(f"Text matrix shape (post-SVD): {text_X.shape}")

# Most "common" terms = lowest IDF (appear in the most documents). Sorting the
# vocabulary by its integer index would just give alphabetical order, not frequency.
feat = np.array(tfidf.get_feature_names_out())
most_common = feat[np.argsort(tfidf.idf_)][:30]
print(f"\n30 most common terms (lowest IDF): {list(most_common)}")


TF-IDF vocabulary size:       41,760 terms
Text matrix shape (post-SVD): (62423, 256)

30 most common terms (lowest IDF): ['the', 'of', 'on', 'bd', 'woman', 'film', 'director', 'and', 'based', 'based on', 'in', 'to', 'woman director', 'movie', 'comedy', 'love', 'book', 'murder', 'relationship', 'nudity', 'new', 'story', 'war', 'of the', 'family', 'dvd', 'from', 'world', 'independent', 'independent film']


## 4. TruncatedSVD — Explained Variance

TruncatedSVD (LSA) compresses the high-dimensional TF-IDF space into a dense
256-dimensional representation. TF-IDF spectra are very flat — text has high intrinsic
dimensionality — so 256 components capture only ~20% of the *total* variance. That is
expected and still yields a useful dense semantic embedding (the alternative, keeping
all 41K sparse dimensions, is intractable for pairwise cosine similarity). The plot
shows the cumulative captured variance.


In [6]:
explained = np.cumsum(svd_transformer.explained_variance_ratio_)

fig = px.line(
    x=list(range(1, len(explained) + 1)), y=explained,
    title="Cumulative Explained Variance — TruncatedSVD on TF-IDF",
    labels={"x": "Number of Components", "y": "Cumulative Explained Variance"},
)
fig.add_annotation(
    x=len(explained), y=explained[-1],
    text=f"{explained[-1]:.1%} at {len(explained)} components",
    showarrow=True, arrowhead=1, ax=-60, ay=-30,
)
save_fig(fig, "07_svd_explained_variance")


def components_for(target):
    idx = int(np.searchsorted(explained, target))
    return f"{idx + 1}" if idx < len(explained) else f"not reached within {len(explained)} comps"

print(f"Components to 90% variance: {components_for(0.90)}")
print(f"Components to 95% variance: {components_for(0.95)}")
print(f"Selected n_components:      {len(explained)}")
print(f"Variance retained at {len(explained)} comps: {explained[-1]:.2%}")


Components to 90% variance: not reached within 256 comps
Components to 95% variance: not reached within 256 comps
Selected n_components:      256
Variance retained at 256 comps: 19.69%


## 5. Assemble & Save Item Feature Matrix

The genre block and the SVD-compressed text block are horizontally stacked
into a single sparse matrix. All fitted transformers are serialised alongside
the matrix so that serving requires no re-fitting.


In [7]:
item_features, movie_index, tfidf_fitted, svd_fitted = build_item_features(movies, n_components=256)

print(f"Item feature matrix shape: {item_features.shape}")
print(f"  Genre block:             {genre_X.shape[1]} dims")
print(f"  Text block (SVD-256):    256 dims")
print(f"Non-zero elements:         {item_features.nnz:,}")
density = item_features.nnz / (item_features.shape[0] * item_features.shape[1])
print(f"Matrix density:            {density:.4%}")

save_item_features(item_features, movie_index, tfidf_fitted, svd_fitted)

print("\nSaved:")
print("  data/processed/item_features.npz")
print("  data/processed/movie_index.parquet")
print("  artifacts/models/tfidf.joblib")
print("  artifacts/models/svd_text.joblib")


Item feature matrix shape: (62423, 276)
  Genre block:             20 dims
  Text block (SVD-256):    256 dims
Non-zero elements:         15,462,835
Matrix density:            89.7502%

Saved:
  data/processed/item_features.npz
  data/processed/movie_index.parquet
  artifacts/models/tfidf.joblib
  artifacts/models/svd_text.joblib


## 6. Feature-Space Sanity Checks

Numbers are abstract — these two checks make the feature space *tangible*.

1. **Nearest neighbours**: for a couple of well-known films, the most content-similar
   movies (by cosine similarity on the 276-dim vectors). If the features are sensible,
   the neighbours should be recognisably related.
2. **2-D projection**: a PCA scatter of a random sample of movies, coloured by their
   primary genre — we expect genres to form visible clusters.


In [8]:
# Dense, L2-normalised copy → cosine similarity is a single matmul (same trick the
# ContentBasedRecommender uses at serving time).
dense = item_features.toarray().astype("float32")
norms = np.linalg.norm(dense, axis=1, keepdims=True)
norms[norms == 0] = 1.0
dn = dense / norms


def most_similar(title, n=5):
    matches = movies.index[movies["clean_title"] == title]
    if len(matches) == 0:
        matches = movies.index[movies["clean_title"].str.contains(title, case=False, na=False)]
    i = int(matches[0])
    sims = dn @ dn[i]
    sims[i] = -1.0
    top = np.argsort(-sims)[:n]
    return (
        movies.iloc[top][["clean_title", "year", "genres"]]
        .assign(similarity=np.round(sims[top], 3))
        .reset_index(drop=True)
    )


for query in ["Toy Story", "Pulp Fiction", "The Matrix"]:
    print(f"Most content-similar to '{query}':")
    display(most_similar(query))


Most content-similar to 'Toy Story':


,clean_title,year,genres,similarity
0,Toy Story 2,1999,Adventure|Animation|Children|Comedy|Fantasy,0.995
1,"Monsters, Inc.",2001,Adventure|Animation|Children|Comedy|Fantasy,0.994
2,"Tale of Despereaux, The",2008,Adventure|Animation|Children|Comedy|Fantasy,0.986
3,Wonder Park,2019,Adventure|Animation|Children|Comedy|Fantasy,0.986
4,Shrek the Third,2007,Adventure|Animation|Children|Comedy|Fantasy,0.986


Most content-similar to 'Pulp Fiction':


,clean_title,year,genres,similarity
0,In Bruges,2008,Comedy|Crime|Drama|Thriller,0.990
1,Fargo,1996,Comedy|Crime|Drama|Thriller,0.987
2,Executive Koala,2005,Comedy|Crime|Drama|Thriller,0.980
3,Leaves of Grass,2009,Comedy|Crime|Drama|Thriller,0.980
4,Dirty Weekend,1973,Comedy|Crime|Drama|Thriller,0.979


Most content-similar to 'The Matrix':


,clean_title,year,genres,similarity
0,Journey to the Edge of the Universe,2008,Documentary,0.978
1,The Medici: Godfathers of the Renaissance,2004,Documentary,0.978
2,The Soul of the Bone,2004,Documentary,0.978
3,Miles Davis: Birth of the Cool,2019,Documentary,0.977
4,Weight of the Nation,2012,Documentary,0.977


In [9]:
from sklearn.decomposition import PCA

rng_viz     = np.random.default_rng(42)
sample_idx  = rng_viz.choice(len(movies), size=min(4000, len(movies)), replace=False)
coords      = PCA(n_components=2, random_state=42).fit_transform(dense[sample_idx])
primary     = movies.iloc[sample_idx]["genres"].str.split("|").str[0].values

viz = pd.DataFrame({"PC1": coords[:, 0], "PC2": coords[:, 1], "Primary genre": primary})
top_genres = viz["Primary genre"].value_counts().head(8).index
viz = viz[viz["Primary genre"].isin(top_genres)]

fig = px.scatter(
    viz, x="PC1", y="PC2", color="Primary genre", opacity=0.6,
    title="Item Feature Space — PCA-2D of 276-dim vectors (4,000-movie sample)",
)
fig.update_traces(marker_size=5)
save_fig(fig, "15_feature_space_pca")


## 7. Non-linear Projections — t-SNE & UMAP

PCA is linear, so it can flatten curved structure. Two non-linear embeddings often
reveal tighter genre clusters. We project the **same 4,000-movie sample** to 2-D,
coloured by primary genre. Both run on a 50-dim PCA pre-reduction — standard practice
that speeds up t-SNE and denoises the input. UMAP needs the optional `umap-learn`
package; the cell degrades gracefully if it isn't installed.

> Runtime: t-SNE on 4,000 points takes ~1–2 minutes; UMAP is faster.


In [10]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# 50-dim PCA pre-reduction shared by both non-linear projections.
X_pca50 = PCA(n_components=50, random_state=42).fit_transform(dense[sample_idx])

tsne_xy = TSNE(
    n_components=2, init="pca", perplexity=30, random_state=42,
).fit_transform(X_pca50)

viz_tsne = pd.DataFrame({"x": tsne_xy[:, 0], "y": tsne_xy[:, 1], "Primary genre": primary})
viz_tsne = viz_tsne[viz_tsne["Primary genre"].isin(top_genres)]

fig = px.scatter(
    viz_tsne, x="x", y="y", color="Primary genre", opacity=0.6,
    title="Item Feature Space — t-SNE (2-D, 4,000-movie sample)",
)
fig.update_traces(marker_size=5)
save_fig(fig, "16_feature_space_tsne")


In [11]:
try:
    import umap  # provided by the `umap-learn` package

    umap_xy = umap.UMAP(
        n_components=2, n_neighbors=15, min_dist=0.1, random_state=42,
    ).fit_transform(X_pca50)

    viz_umap = pd.DataFrame({"x": umap_xy[:, 0], "y": umap_xy[:, 1], "Primary genre": primary})
    viz_umap = viz_umap[viz_umap["Primary genre"].isin(top_genres)]

    fig = px.scatter(
        viz_umap, x="x", y="y", color="Primary genre", opacity=0.6,
        title="Item Feature Space — UMAP (2-D, 4,000-movie sample)",
    )
    fig.update_traces(marker_size=5)
    save_fig(fig, "17_feature_space_umap")
except ImportError:
    print("umap-learn not installed — skipping UMAP. Install with:  pip install umap-learn")


## Conclusion

- The item feature matrix has shape **(n_movies × 276)**: 20 genre indicator dimensions plus 256 SVD-compressed text dimensions.
- The TF-IDF vocabulary spans thousands of unique terms (min_df = 5, bigrams), capturing both title semantics and community tag vocabulary.
- **256 SVD components** capture ~20% of the (very flat) TF-IDF variance — typical for sparse text — while giving a compact dense embedding that keeps pairwise cosine similarity tractable across 62K items.
- Genre features are directly interpretable and complement the semantic smoothing provided by SVD text features.
- All transformers are serialised to `artifacts/models/`, ensuring the exact same feature space is applied at serving time without re-fitting.
